# Notebook 2 — The SLC26 Superfamily: From Tree to Prestin

## Learning objectives
- Build a phylogenetic tree with **FastTree**
- Annotate and explore trees interactively with **ETE4 smartview**
- Distinguish **ortholog groups** (subfamilies) from **duplication events**
- Extract the **prestin (SLC26A5)** subfamily
- Compare tree topologies from **different methods** (FastTree, IQ-TREE, NJ)
- Understand why method choice matters for detecting convergent evolution

## Recap
From Notebook 1 we have a trimmed alignment of 297 SLC26 protein sequences from 30 mammalian species. Now we build the tree and look inside.

In [ ]:
import os, subprocess, io
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from Bio import SeqIO, AlignIO

DATA = os.path.join("..", "data")
FIGS = os.path.join("..", "figures")
FASTA = os.path.join(DATA, "selection2_clustalo.fa")

# Trimmed alignment from Notebook 1
TRIM = os.path.join(DATA, "slc26.mafft.gt01.fa")
if not os.path.exists(TRIM):
    # If Notebook 1 wasn't run, build it now
    ALN = os.path.join(DATA, "slc26.mafft.fa")
    if not os.path.exists(ALN):
        print("Running MAFFT...")
        with open(ALN, "w") as f:
            subprocess.run(["mafft", "--auto", "--thread", "-1", FASTA],
                           stdout=f, stderr=subprocess.PIPE, check=True)
    subprocess.run(["trimal", "-in", ALN, "-out", TRIM, "-gt", "0.1", "-fasta"],
                   check=True, capture_output=True)
    print(f"Built: {TRIM}")

aln_trim = AlignIO.read(TRIM, "fasta")
print(f"Trimmed alignment: {len(aln_trim)} sequences × {aln_trim.get_alignment_length()} columns")

---
## 1. Tree inference with FastTree

[FastTree](http://www.microbesonline.org/fasttree/) builds approximately-maximum-likelihood trees. It's fast enough for hundreds of sequences and uses the **LG** amino acid substitution model (`-lg`). For more thorough ML inference, **IQ-TREE** is the standard — we'll compare later.

In [ ]:
TREE = os.path.join(DATA, "slc26.mafft.gt01.fasttree.nwk")

if not os.path.exists(TREE):
    print("Running FastTree (LG model)...")
    with open(TREE, "w") as f:
        subprocess.run(["FastTree", "-lg", TRIM], stdout=f,
                       stderr=subprocess.PIPE, check=True)
    print("Done.")
else:
    print(f"Cached: {TREE}")

---
## 2. Loading the tree in ETE4

[ETE4](https://etetoolkit.github.io/ete/) extends ETE3 with an interactive browser-based viewer (**smartview**). Its `PhyloTree` can annotate NCBI taxonomy and infer duplication/speciation events, just like ETE3 — but with a much richer interactive interface.

In [ ]:
from ete4 import PhyloTree
from ete4.smartview import Layout, TextFace, SeqFace

# Load tree — taxid is the part before the dot
t = PhyloTree(open(TREE).read(),
              sp_naming_function=lambda node: node.name.split('.')[0])

# Root at midpoint
t.set_outgroup(t.get_midpoint_outgroup())
t.resolve_polytomy(descendants=True)

# Annotate with NCBI taxonomy
print("Annotating with NCBI taxonomy...")
t.annotate_ncbi_taxa()

# Infer duplication/speciation events
events = t.get_descendant_evol_events()

n_dup = sum(1 for e in events if e.etype == "D")
n_spec = sum(1 for e in events if e.etype == "S")
print(f"Duplication events: {n_dup}")
print(f"Speciation events:  {n_spec}")
print(f"Leaves: {len(list(t.leaves()))}") 

### 2.1 What species do we have?

In [ ]:
species_names = {}
for leaf in t.leaves():
    taxid = leaf.name.split(".")[0]
    sci = leaf.props.get("sci_name", taxid)
    species_names[taxid] = sci

taxid_counts = Counter(leaf.name.split(".")[0] for leaf in t.leaves())

print(f"{'TaxID':<10s} {'Species':<40s} {'Genes':>5s}")
print("-" * 60)
for taxid, count in sorted(taxid_counts.items(), key=lambda x: -x[1]):
    sci = species_names.get(taxid, "?")
    print(f"  {taxid:<10s} {sci:<40s} {count:>5d}")

### ✏️ Exercise 5

From the species list above:
1. Which are **bats** (order Chiroptera)? Which are echolocating, which are fruit bats (Pteropodidae)?
2. Which are **cetaceans** (Cetacea)? Toothed whales (Odontoceti, echolocating) vs baleen whales (Mysticeti)?
3. Most species have ~10–11 SLC26 genes. Which have fewer? Why might that be?

**Hint:** All Chiroptera except Pteropodidae echolocate. All Odontoceti echolocate; Mysticeti do not.

Use `NCBITaxa` from Notebook 1 if you need to look up lineages:
```python
from ete3 import NCBITaxa
ncbi = NCBITaxa()
lineage = ncbi.get_lineage(9739)  # dolphin
names = ncbi.get_taxid_translator(lineage)
print([names[t] for t in lineage])
```

---
## 3. Interactive tree exploration (ETE4 smartview)

ETE4's smartview runs in your browser. We define **layouts** — functions that control how each node is drawn.

In [ ]:
# Layout: show accession, gene name, and scientific name
def node_names_style(node):
    if node.is_leaf:
        acc = node.name.split(".")[1] if "." in node.name else node.name
        sci = node.props.get("sci_name", "")
        return [
            TextFace(acc, style={"fill": "black"}, column=0, position="right"),
            TextFace(f"({sci})", style={"fill": "gray"}, column=1, position="right"),
        ]

# Layout: mark duplications with red triangles
def node_evo_style(node):
    if "evoltype" in node.props and node.props["evoltype"] == "D":
        return {"dot": {"shape": "triangle", "radius": 8, "fill": "red"}}

# Layout: tree style tweaks
def tree_modify_style(tree_style):
    tree_style.collapse_size = 1
    tree_style.smartzoom = False

In [ ]:
# Launch interactive viewer
print("Launching ETE4 smartview in your browser...")
print("Look for:")
print("  - Red triangles at internal nodes = gene DUPLICATION events")
print("  - These mark the boundaries between paralog groups (subfamilies)")
print("  - Within each subfamily, nodes are connected by speciation events")

layouts = [
    Layout(name="Tree", draw_tree=tree_modify_style, active=True),
    Layout(name="Names", draw_node=node_names_style),
    Layout(name="Duplications", draw_node=node_evo_style, active=True),
]

t.explore(layouts=layouts, keep_server=True, port=5006,
          show_leaf_name=False,
          include_props=("name", "sci_name", "evoltype", "dist", "support"))

### Orthology and paralogy in the SLC26 tree

In the tree, **red triangles** mark **duplication events** — the ancient gene duplications that created the SLC26 subfamilies (A1–A11) before the mammalian radiation.

**Within** each subfamily clade: all genes are **orthologs** (same gene, different species), connected by speciation events.

**Between** subfamilies: genes are **paralogs** (different genes that arose by duplication in an ancestral genome).

### ✏️ Exercise 6

In the interactive tree:
1. Count the major duplication events. Do you see ~10 paralog clades corresponding to SLC26A1–A11?
2. Which subfamilies are **sister clades** (most closely related)?
3. Can you identify the **prestin (SLC26A5)** clade? (Hint: look for a clade of ~30 leaves — one per species.)
4. Are there any **lineage-specific duplications** within a subfamily?

### ✏️ Exercise 7

Write code to list all leaves in each major subfamily clade.

Hint: iterate over internal nodes, find those with `evoltype == 'D'`, and examine their children sub-clades.

In [ ]:
# YOUR CODE HERE
# Hint:
# for node in t.traverse():
#     if node.props.get("evoltype") == "D":
#         children = node.children
#         for child in children:
#             leaves = [l.name for l in child.leaves()]
#             print(f"Clade ({len(leaves)} leaves): {leaves[:3]}...")


---
## 4. Extracting the prestin (SLC26A5) subfamily

We identify the prestin clade using known accessions, then extract its sequences as a separate FASTA for re-alignment and method comparison.

In [ ]:
# Find human prestin (Q7LBE3) or mouse prestin (Q99NH7)
known_prestin_accessions = {"Q7LBE3", "Q99NH7"}

prestin_leaves = []
for leaf in t.leaves():
    acc = leaf.name.split(".")[1] if "." in leaf.name else leaf.name
    if acc in known_prestin_accessions:
        prestin_leaves.append(leaf)
        print(f"Found: {leaf.name} ({leaf.props.get('sci_name', '?')})")

if prestin_leaves:
    # Get the common ancestor of known prestin sequences
    if len(prestin_leaves) >= 2:
        prestin_ancestor = t.common_ancestor([l.name for l in prestin_leaves])
    else:
        prestin_ancestor = prestin_leaves[0].up

    # Walk up until we hit a duplication node (subfamily boundary)
    node = prestin_ancestor
    while node.up:
        if node.up.props.get("evoltype") == "D":
            break
        node = node.up

    prestin_clade_names = [l.name for l in node.leaves()]
    print(f"\nPrestin clade: {len(prestin_clade_names)} sequences")
else:
    print("⚠ Could not find prestin automatically.")
    print("  Set prestin_clade_names manually from the interactive tree.")
    prestin_clade_names = []

In [ ]:
# Write prestin sequences to a separate FASTA
all_seqs = {rec.id: rec for rec in SeqIO.parse(FASTA, "fasta")}

SUB_DIR = os.path.join(DATA, "subfamilies")
os.makedirs(SUB_DIR, exist_ok=True)

prestin_fasta = os.path.join(SUB_DIR, "prestin.fa")
prestin_recs = [all_seqs[name] for name in prestin_clade_names if name in all_seqs]
SeqIO.write(prestin_recs, prestin_fasta, "fasta")
print(f"Written: {len(prestin_recs)} sequences → {prestin_fasta}")

print("\nSpecies in prestin clade:")
for rec in prestin_recs:
    taxid = rec.id.split(".")[0]
    leaf_matches = list(t.search_nodes(name=rec.id))
    sci = leaf_matches[0].props.get("sci_name", taxid) if leaf_matches else taxid
    print(f"  {rec.id:<25s} {sci}")

### ✏️ Exercise 8

Extract at least one **control subfamily** — e.g., the clade containing human SLC26A3 (DRA, accession P40879) or SLC26A6 (PAT1, accession Q86WA9). These have no role in hearing and serve as negative controls for convergence.

Use the same approach: find a known accession → walk up to the duplication boundary → extract leaves.

In [ ]:
# YOUR CODE HERE


---
## 5. Comparing phylogenetic methods on prestin

Different tree-building methods have different strengths and weaknesses. We test three approaches on the prestin subfamily:

| Method | Type | Notes |
|:---|:---|:---|
| **MAFFT + FastTree** | FFT alignment + approx. ML | Fast, good default |
| **MAFFT + IQ-TREE** | FFT alignment + full ML + ModelFinder | Best-fit model, ultrafast bootstrap |
| **MAFFT + NJ** | FFT alignment + distance-based | Fastest, but most susceptible to convergence |

**Why this matters:** distance-based methods (NJ) are more susceptible to long-branch attraction and convergent substitutions. If echolocation drives convergent amino acid changes in prestin, NJ may group echolocators together even though they are not closely related.

In [ ]:
def build_prestin_tree(fasta_in, prefix, tree_method="fasttree", trim_gt="0.1"):
    """Align with MAFFT → trim → build tree. Returns (tree_path, trim_path)."""
    aln_f = f"{prefix}.mafft.fa"
    trim_f = f"{prefix}.mafft.gt{trim_gt.replace('.','')}.fa"
    tree_f = f"{prefix}.mafft.{tree_method}.nwk"

    # Align
    if not os.path.exists(aln_f):
        with open(aln_f, "w") as f:
            subprocess.run(["mafft", "--auto", "--thread", "-1", fasta_in],
                           stdout=f, stderr=subprocess.PIPE, check=True)

    # Trim
    if not os.path.exists(trim_f):
        subprocess.run(["trimal", "-in", aln_f, "-out", trim_f,
                         "-gt", trim_gt, "-fasta"],
                       check=True, capture_output=True)

    # Tree
    if not os.path.exists(tree_f):
        if tree_method == "fasttree":
            with open(tree_f, "w") as f:
                subprocess.run(["FastTree", "-lg", trim_f],
                               stdout=f, stderr=subprocess.PIPE, check=True)

        elif tree_method == "iqtree":
            pfx = tree_f.replace(".nwk", "")
            subprocess.run(["iqtree2", "-s", trim_f, "--prefix", pfx,
                            "-mset", "LG", "-B", "1000", "-T", "AUTO", "--quiet"],
                           check=True, capture_output=True)
            if os.path.exists(pfx + ".treefile"):
                os.rename(pfx + ".treefile", tree_f)

        elif tree_method == "nj":
            from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
            from Bio import Phylo
            a = AlignIO.read(trim_f, "fasta")
            dm = DistanceCalculator("blosum62").get_distance(a)
            nj_tree = DistanceTreeConstructor().nj(dm)
            buf = io.StringIO()
            Phylo.write(nj_tree, buf, "newick")
            with open(tree_f, "w") as f:
                f.write(buf.getvalue())

    a = AlignIO.read(trim_f, "fasta")
    print(f"  {tree_method}: {len(a)} seqs × {a.get_alignment_length()} cols → {os.path.basename(tree_f)}")
    return tree_f, trim_f

In [ ]:
# Build prestin trees with three methods
prestin_trees = {}
for method in ["fasttree", "iqtree", "nj"]:
    prefix = os.path.join(SUB_DIR, f"prestin_{method}")
    try:
        tree_f, trim_f = build_prestin_tree(prestin_fasta, prefix, tree_method=method)
        prestin_trees[method] = {"tree": tree_f, "trim": trim_f}
    except Exception as e:
        print(f"  ⚠ {method}: {e}")

### 5.1 Annotate and compare trees

We highlight echolocating species in each tree. **You need to fill in the echolocating taxids** based on your species classification from Exercise 5.

In [ ]:
# FILL IN based on your species classification (Exercise 5)
# Toothed whales (Odontoceti) — these definitely echolocate
# Echolocating bats — all Chiroptera EXCEPT Pteropodidae (fruit bats)
ECHOLOCATING_TAXIDS = {
    # Toothed whales
    "9739",     # Tursiops truncatus (bottlenose dolphin)
    # Add more cetacean taxids...
    # Echolocating bats — add from your species list
    # e.g., "59479",  # Rhinolophus ferrumequinum
    # ...
}

if len(ECHOLOCATING_TAXIDS) < 5:
    print("⚠ You need to complete ECHOLOCATING_TAXIDS!")
    print("  Use NCBITaxa to look up which species are bats/cetaceans.")
    print("  All Chiroptera except Pteropodidae echolocate.")
    print("  All Odontoceti echolocate.")

In [ ]:
# Explore each tree with echolocators highlighted
for method, paths in prestin_trees.items():
    print(f"\n{'='*50}")
    print(f"  Prestin — {method}")
    print(f"{'='*50}")

    pt = PhyloTree(open(paths["tree"]).read(),
                   sp_naming_function=lambda n: n.split('.')[0])
    pt.set_outgroup(pt.get_midpoint_outgroup())
    pt.annotate_ncbi_taxa()

    def echo_layout(node):
        if not node.is_leaf:
            return {}
        taxid = node.name.split(".")[0]
        is_echo = taxid in ECHOLOCATING_TAXIDS
        sci = node.props.get("sci_name", taxid)
        color = "#D85A30" if is_echo else "#888888"
        weight = "bold" if is_echo else "normal"
        return [
            TextFace(f"{sci}", style={"fill": color, "font-weight": weight},
                     column=0, position="right"),
        ]

    port = 5010 + list(prestin_trees.keys()).index(method)
    layouts = [Layout(name="Echolocators", draw_node=echo_layout)]

    pt.explore(layouts=layouts, keep_server=True, port=port,
               show_leaf_name=False, name=f"Prestin — {method}",
               include_props=("name", "sci_name", "dist", "support"))

### ✏️ Exercise 9

Compare the three prestin trees:
1. In which trees do echolocating bats cluster with the **dolphin**?
2. In which trees do **Yinpterochiroptera** bats (*Rhinolophus*, *Hipposideros*) cluster with **Yangochiroptera** (*Myotis*, *Eptesicus*)?
3. Which method is **most susceptible** to convergent grouping? Why?
4. Where do **fruit bats** (Pteropodidae, non-echolocating) fall?

### ✏️ Exercise 10

Build the same set of trees for your **control subfamily** (from Exercise 8). Do echolocators cluster together in those trees? If convergence is specific to prestin, the control should recover the species tree.

In [ ]:
# YOUR CODE HERE


---
## 6. Alignment visualization on the tree (SeqFace)

ETE4 can display the alignment alongside the tree, letting you visually spot columns where echolocators share a residue that differs from the rest.

In [ ]:
# Attach alignment sequences to the FastTree prestin tree
if "fasttree" in prestin_trees:
    trim_path = prestin_trees["fasttree"]["trim"]
    tree_path = prestin_trees["fasttree"]["tree"]

    name2seq = {rec.id: str(rec.seq) for rec in SeqIO.parse(trim_path, "fasta")}

    pt = PhyloTree(open(tree_path).read(),
                   sp_naming_function=lambda n: n.split('.')[0])
    pt.set_outgroup(pt.get_midpoint_outgroup())
    pt.annotate_ncbi_taxa()

    for leaf in pt.leaves():
        if leaf.name in name2seq:
            leaf.add_prop("seq", name2seq[leaf.name])

    def layout_seq(node):
        if node.is_leaf:
            seq = node.props.get("seq")
            if seq:
                return [SeqFace(seq, seqtype="aa", poswidth=15, position="aligned")]

    layouts = [
        Layout(name="Alignment", draw_node=layout_seq, active=True),
        Layout(name="Echolocators", draw_node=echo_layout),
    ]

    print("Launching tree + alignment view...")
    print("Look for columns where echolocators share a residue")
    print("that differs from non-echolocators.")

    pt.explore(layouts=layouts, keep_server=True, port=5020,
               show_leaf_name=False, name="Prestin + alignment")

---
## Summary

- Built the SLC26 family tree with **FastTree** (LG model)
- Used **ETE4 smartview** to explore the tree interactively
- Identified **~10 subfamily clades** separated by ancient duplication events
- Extracted the **prestin (SLC26A5)** subfamily
- Compared **FastTree**, **IQ-TREE**, and **NJ** trees for prestin
- Observed that **NJ is most susceptible** to convergent signal — echolocators tend to cluster
- Visualized the **alignment alongside the tree** with SeqFace

**Tomorrow (Day 2):** We systematically detect convergent residues, test significance with permutation tests, compare prestin to control subfamilies, and map convergent sites onto the 3D structure.